In [16]:
with open('DATA.txt','r',encoding='utf-8') as f:
    text = f.read()

In [17]:
print(f'length of dataset in characters: {len(text)}')

length of dataset in characters: 1115393


In [18]:
#first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [19]:
chars =sorted(list(set(text)))
num_chars =len(chars)
print(f'number of unique characters: {num_chars}')
print(''.join(chars))

number of unique characters: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [21]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hello world'))
print(decode(encode('hello world')))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


In [22]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [23]:
n = int(0.9*len(data)) #train data 90% 
train_data = data[:n] #first 90% of data for training rest val data
val_data = data[n:]
print(train_data.shape, val_data.shape) 

torch.Size([1003853]) torch.Size([111540])


In [24]:
block_size = 8 
train_data[:block_size+1] #first block_size+1 characters from train data

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [25]:
x = train_data[:block_size] #first block_size characters from train data
y = train_data[1:block_size+1] #next block_size characters from train data
print('inputs: ', x)
print('targets: ', y)
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f'when input is {context} the target: {target}')
    

inputs:  tensor([18, 47, 56, 57, 58,  1, 15, 47])
targets:  tensor([47, 56, 57, 58,  1, 15, 47, 58])
when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [26]:
torch.manual_seed(1337) #for reproducibility
batch_size = 4 
block_size = 8

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y 

xb , yb = get_batch('train')
print('inputs: ')
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)
print('=============================')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f'when input is {context} the target: {target}')

inputs: 
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets: 
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
when input is tensor([53]) the target: 59
when input is tensor([53, 59]) the target: 6
when input is tensor([53, 59,  6]) the target: 1
when input is tensor([53, 59,  6,  1]) the target: 58
when input is tensor([53, 59,  6,  1, 58]) the target: 56
when input is tensor([53, 59,  6,  1, 58, 56]) the target: 47
when input is tensor([53, 59,  6,  1, 58, 56, 47]) the target: 40
when input is tensor([53, 59,  6,  1, 58, 56, 47, 40]) the target: 59
when input is tensor([49]) the target: 43
when input is tensor([49, 43]) the target: 43
when input is tensor([49, 43, 43]) the target: 54
when input is tensor(

In [35]:
#bigram model
import torch 
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

vocab_size = num_chars

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logist for the next token form the lookup table 
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets = None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) batch time chennel 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
        return logits, loss 

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # get the predictions 
            logits = logits[:, -1, :] # becomes (B,C) the last time step 
            probs = F.softmax(logits, dim=-1) # (B,C) 
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1) sample from the distribution 
            idx = torch.cat((idx, idx_next), dim=1) # append sampled index to the running sequence (B,T+1)
            
        return idx
    
    
m = BigramLanguageModel(vocab_size)

logits, loss = m(xb, yb)

print(logits.shape) #(batch_size, block_size, vocab_size)
print(loss) #cross entropy loss

torch.Size([32, 65])
tensor(4.8948, grad_fn=<NllLossBackward0>)


In [36]:
 print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [37]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [43]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    print(f'step {steps}: loss {loss.item()}')

step 0: loss 2.7888166904449463
step 1: loss 2.741046190261841
step 2: loss 2.780517101287842
step 3: loss 2.742469549179077
step 4: loss 2.700695514678955
step 5: loss 2.7823026180267334
step 6: loss 2.6685848236083984
step 7: loss 2.696444034576416
step 8: loss 2.7556920051574707
step 9: loss 2.835480213165283
step 10: loss 2.6747612953186035
step 11: loss 2.690251588821411
step 12: loss 2.759465456008911
step 13: loss 2.754635810852051
step 14: loss 2.76943039894104
step 15: loss 2.728788375854492
step 16: loss 2.683819532394409
step 17: loss 2.784728765487671
step 18: loss 2.728734254837036
step 19: loss 2.7420663833618164
step 20: loss 2.7489423751831055
step 21: loss 2.8025341033935547
step 22: loss 2.738335609436035
step 23: loss 2.754317283630371
step 24: loss 2.700056314468384
step 25: loss 2.753237009048462
step 26: loss 2.8240463733673096
step 27: loss 2.731985569000244
step 28: loss 2.7957069873809814
step 29: loss 2.7249090671539307
step 30: loss 2.7168691158294678
step 31

In [44]:
 print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


SSimathoo I touis thasely End, ory paf nd?
I lles hieltints feeacubldapio ny d,
An wald
FLingacre
Ther my
No isalon stary RKEOnores us w be g? bagof tts! at ame owo'd bandre,
JUMonono trawnthetrid ang pat
My t nco grwinste t w rtce,

NIDULIRGorr best gerer, nt; co hamyose EESof bun 'APAExcowhe;
UCHAnd d bumod
Ghacendvevesou mbouk'su mbam

S:
OMIUCoake:
them man.
BEYOMETht l at
Sun st tite wit we t mpt mamas lissherrd tond murd sug, wrode.
Thackinonour'se Per s wanothepaf-
AUFowaid anixat:
KEDWas


In [49]:
# toy example 

torch.manual_seed(1337)

B,T,C = 4, 8, 2 
X = torch.randn(B,T,C)
X.shape

torch.Size([4, 8, 2])

In [51]:
#we want x[b,t] == torch.zeros((B,T,C))
xBOW = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xpev = X[b,:t+1] # (t,c)
        xBOW[b,t] = torch.mean(xpev, dim=0) # (c)

In [63]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)

xBOW2 = wei @ X # (B,T,T) @ (B,T,C) --> (B,T,C)
xBOW2
print(torch.allclose(xBOW, xBOW2))


False


In [ ]:
# ver 3 using softmax 

tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))  
wei = torch.softmax(wei, dim=-1)
xBOW3 = wei @ X # (B,T,T) @ (B,T,C) --> (B,T,C)
print(torch.allclose(xBOW, xBOW3))

False


In [54]:
X[0]
xBOW[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [73]:
# version 4 self attention:

torch.manual_seed(1337)
B,T,C = 4, 8, 32 
X = torch.randn(B,T,C)

#lets try to get the self attention weights for single head
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False) 
k = key(X) # (B,T,16) 
q = query(X) # (B,T,16)
wei = q @k.transpose(-2,-1) # (B,T,16) @ (B,16,T) --> (B,T,T)

tril = torch.tril(torch.ones(T,T))
# print(tril)
# wei = torch.zeros((T,T))
# print(wei)
wei = wei.masked_fill(tril == 0, float('-inf'))
# # print(wei)  
wei = torch.softmax(wei, dim=-1)
print(wei[0])
v = value(X) # (B,T,16)
out = wei @ v # (B,T,T) @ (B,T,C) --> (B,T,C)
print(out.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
torch.Size([4, 8, 16])


In [55]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, dim=1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b 
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)


a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
